# EXP-07 — Comparação exploratória controlada de quatro classificadores

**Prazo do artigo: 20/11/2026.** Este notebook **não reetiqueta** os 9 alertas de 2026, nem usa os dados de 2020/2026 como conjunto rotulado. As métricas de validação interna **não são desempenho em protocolos novos**. Caso a validação por contrato não seja possível, o notebook deixa isso explícito.

**Modelos:** Random Forest (100 árvores, profundidade 5, correspondente à configuração de referência), Regressão Logística, Árvore de Decisão, Gradient Boosting. As cinco características numéricas são fixas; endereço e dia são usados para auditoria e separação, nunca como preditores. A escolha do limiar 0,5 é fixa; sem otimização.


In [ ]:
%pip -q install pyarrow scikit-learn pandas google-api-python-client


## 1. Baixar a matriz do Drive oficial
Usa a sessão autenticada da sua conta Google. **Não cole tokens nem URLs privadas de RPC.**

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from datetime import datetime, timezone
import io, os, hashlib, json

auth.authenticate_user()
service=build('drive','v3')
MATRIX_ID='1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S'
ROOT=Path('/content/EXP-07')
DEST=ROOT/('EXP-07_comparativo_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
DEST.mkdir(parents=True,exist_ok=False)
req=service.files().get_media(fileId=MATRIX_ID)
buf=io.BytesIO();downloader=MediaIoBaseDownload(buf,req)
done=False
while not done:
    _,done=downloader.next_chunk()
MATRIX=DEST/'features_treino_original.parquet';MATRIX.write_bytes(buf.getvalue())
H=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
print('Matriz bytes:',MATRIX.stat().st_size,'SHA256:',H(MATRIX),'Saída:',DEST)


## 2. Auditoria obrigatória antes de qualquer treinamento
Verificamos classe, número de linhas e diversidade dos contratos. Se apenas um contrato positivo estiver presente, **não existe avaliação válida de generalização por contrato**.

In [ ]:
import numpy as np, pandas as pd
D=pd.read_parquet(MATRIX)
FEATURES=['n_tx','value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
REQUIRED=['recipient','dia','label']+FEATURES
missing=[k for k in REQUIRED if k not in D.columns]
if missing:raise ValueError('Colunas obrigatórias ausentes: '+str(missing))
assert D['label'].dropna().isin([0,1]).all(), 'Rótulos não binários; revisar arquivo'
assert D['label'].notna().all(), 'Rótulos vazios; interromper'
D['recipient']=D['recipient'].astype(str).str.lower()
D['dia']=pd.to_datetime(D['dia'],errors='coerce')
assert not D['dia'].isna().any(), 'Datas inválidas; interromper'
assert D['recipient'].str.startswith('0x').all(), 'Endereço inesperado'
print('Formato:',D.shape)
print('Colunas:',list(D.columns))
print('Classes:',D['label'].value_counts().sort_index().to_dict())
print('Contratos por classe:',D.groupby('label')['recipient'].nunique().to_dict())
print('Amostras contrato-dia duplicadas:',int(D.duplicated(['recipient','dia']).sum()))
print('Dias por classe:',D.groupby('label')['dia'].nunique().to_dict())
print('Endereços positivos (apenas auditoria, não fornecer aos revisores EXP-05):',D.loc[D.label==1,'recipient'].nunique())
assert set(D.label)=={0,1},'É necessário haver ambas as classes'
for col in FEATURES:D[col]=pd.to_numeric(D[col],errors='coerce')
summary={'n_rows':len(D),'classes':{str(k):int(v) for k,v in D.label.value_counts().items()},'contracts_by_class':{str(k):int(v) for k,v in D.groupby('label').recipient.nunique().items()},'duplicate_contract_day':int(D.duplicated(['recipient','dia']).sum()),'features':FEATURES,'matrix_sha256':H(MATRIX)}
(DEST/'auditoria_matriz.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')


## 3. Validação conservadora e comparação
Validação agrupada por endereço só ocorre se houver ao menos **dois contratos positivos independentes e dois negativos**, além de dois splits com ambas as classes. Caso contrário, executamos apenas **validação estratificada exploratória entre linhas**, explicitamente sujeita a vazamento de identidade. **Não publicar F1 desse caso como desempenho de detecção em contratos novos.**

Não inclua endereço nem data nas features. O pré-processamento pertence ao `Pipeline` em cada dobra, evitando vazamento de imputação/escala.

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix

X=D[FEATURES]; y=D['label'].astype(int);groups=D['recipient']; seed=20261119
models={
'RandomForest':Pipeline([('imputer',SimpleImputer(strategy='median')),('clf',RandomForestClassifier(n_estimators=100,max_depth=5,random_state=seed,n_jobs=1))]),
'RegressaoLogistica':Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('clf',LogisticRegression(max_iter=2000,random_state=seed,class_weight='balanced'))]),
'DecisionTree':Pipeline([('imputer',SimpleImputer(strategy='median')),('clf',DecisionTreeClassifier(max_depth=5,random_state=seed))]),
'GradientBoosting':Pipeline([('imputer',SimpleImputer(strategy='median')),('clf',GradientBoostingClassifier(random_state=seed))])}

def valid_splits(splitter,grouped=False):
    pairs=list(splitter.split(X,y,groups if grouped else None))
    for tr,te in pairs:
        if set(y.iloc[tr])!={0,1} or set(y.iloc[te])!={0,1}:return None
        if grouped and set(groups.iloc[tr])&set(groups.iloc[te]):return None
    return pairs

contract_counts=D.groupby('label').recipient.nunique()
can_group=all(contract_counts.get(k,0)>=2 for k in (0,1))
splits=None
if can_group:
    try:splits=valid_splits(StratifiedGroupKFold(n_splits=2,shuffle=True,random_state=seed),True)
    except ValueError:splits=None
if splits is not None:
    method='estratificada_agrupada_por_contrato_2_folds';validated_external=False
else:
    minclass=int(y.value_counts().min())
    if minclass<2:raise ValueError('Classe com menos de 2 exemplos: comparação não executável')
    nsplit=min(5,minclass)
    splits=valid_splits(StratifiedKFold(n_splits=nsplit,shuffle=True,random_state=seed))
    method='estratificada_por_linha_EXPLORATORIA_SEM_VALIDACAO_EXTERNA'
    validated_external=False
print('Estratégia:',method,'N folds:',len(splits))
rows=[]
for model_name,model in models.items():
    for i,(tr,te) in enumerate(splits,1):
        model.fit(X.iloc[tr],y.iloc[tr]);pred=model.predict(X.iloc[te])
        cm=confusion_matrix(y.iloc[te],pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
        rows.append({'modelo':model_name,'fold':i,'n_treino':len(tr),'n_teste':len(te),'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),'precision':precision_score(y.iloc[te],pred,zero_division=0),'recall':recall_score(y.iloc[te],pred,zero_division=0),'f1':f1_score(y.iloc[te],pred,zero_division=0),'balanced_accuracy':balanced_accuracy_score(y.iloc[te],pred),'estrategia':method})
R=pd.DataFrame(rows);R.to_csv(DEST/'metricas_por_fold.csv',index=False)
S=R.groupby('modelo')[['precision','recall','f1','balanced_accuracy']].agg(['mean','std']).round(4)
print(S.to_string());S.to_csv(DEST/'resumo_modelos.csv')
if 'EXPLORATORIA' in method:print('ATENÇÃO: esta validação não mede generalização para novos contratos/protocolos.')


## 4. Inspecionar distribuição dos alertas em 2026 (opcional)
**Não serão calculadas precision/recall/F1** para 2026, pois contratos sem rótulos independentes não são negativos confirmados. Para evitar falsa interpretação, esta versão **não reproduz contagens em 2026**; apenas compara modelos dentro do dataset de treinamento.

In [ ]:
manifest={'experimento':'EXP-07','gerado_utc':datetime.now(timezone.utc).isoformat(),'tipo':'comparacao_exploratoria','dataset_matriz_id':MATRIX_ID,'metodo_validacao':method,'grupos_positivos_independentes':int(contract_counts.get(1,0)),'grupo_por_contrato_validado':bool('agrupada' in method),'generalizacao_para_protocolo_novo_demonstrada':False,'features':FEATURES,'limites':['Amostras positivas podem vir de um unico contrato','Validacao por linhas pode vazar identidade e temporalidade','2020 e 2026 nao usados como teste rotulado','Nao usar este resultado para afirmar novo F1 populacional','Sem conclusao de superioridade por variacao entre folds'],'sha256':{p.name:H(p) for p in sorted(DEST.iterdir()) if p.is_file()}}
(DEST/'manifesto_exp07.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
parent='1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI' # pasta oficial results
folder=service.files().create(body={'name':DEST.name,'mimeType':'application/vnd.google-apps.folder','parents':[parent]},fields='id,name').execute()
for p in sorted(DEST.iterdir()):
    if p.is_file():
        obj=service.files().create(body={'name':p.name,'parents':[folder['id']]},media_body=MediaFileUpload(str(p),mimetype='application/octet-stream',resumable=True),fields='id,name').execute()
        print('Salvo:',obj['name'])
print('RESULTADOS: https://drive.google.com/drive/folders/'+folder['id'])
print('Método:',method)
